# LC 74 — Search a 2D Matrix
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Binary Search
**Pattern:** Flatten 2D to 1D — Binary Search with Index Mapping

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> The matrix is one sorted
list wrapped across rows. Treat it as a 1D array:
mid // cols gives the row, mid % cols gives the
column. Binary search on the virtual 1D index.
</div>

## Official Problem Statement

You are given an `m x n` integer matrix `matrix`
with the following two properties:

- Each row is sorted in non-decreasing order.
- The first integer of each row is greater than
  the last integer of the previous row.

Given an integer `target`, return `true` if
`target` is in `matrix` or `false` otherwise.

You must write a solution in `O(log(m * n))` time
complexity.

**Example 1:**
```
Input:  matrix = [[1,3,5,7],[10,11,16,20],[23,30,34,60]]
        target = 3
Output: true
```
**Example 2:**
```
Input:  matrix = [[1,3,5,7],[10,11,16,20],[23,30,34,60]]
        target = 13
Output: false
```

**Constraints:**
- `m == matrix.length`
- `n == matrix[0].length`
- `1 <= m, n <= 100`
- `-10^4 <= matrix[i][j], target <= 10^4`

## What This Is Actually Asking

The matrix is like a sorted list folded into rows:
each row picks up exactly where the last row ended.
Check whether a number exists anywhere in this grid.
You must do it in O(log(m×n)) — one binary search,
not a binary search per row.

## Walk Through an Example by Hand

```
matrix = [[1, 3, 5, 7],
           [10,11,16,20],
           [23,30,34,60]]
target = 3   rows=3  cols=4   total=12

lo=0  hi=11  (0 to rows*cols-1)

Step 1:
  mid = (0+11)//2 = 5
  row = 5//4 = 1   col = 5%4 = 1
  matrix[1][1] = 11   11 > 3 -> hi = mid-1 = 4

Step 2:
  mid = (0+4)//2 = 2
  row = 2//4 = 0   col = 2%4 = 2
  matrix[0][2] = 5   5 > 3 -> hi = mid-1 = 1

Step 3:
  mid = (0+1)//2 = 0
  row = 0//4 = 0   col = 0%4 = 0
  matrix[0][0] = 1   1 < 3 -> lo = mid+1 = 1

Step 4:
  mid = (1+1)//2 = 1
  row = 1//4 = 0   col = 1%4 = 1
  matrix[0][1] = 3   3 == 3 -> return True
```

## The Picture

```
matrix = [[ 1,  3,  5,  7],
           [10, 11, 16, 20],
           [23, 30, 34, 60]]

Unfold into one sorted list:
  idx:  0   1   2   3   4   5   6   7   8   9  10  11
  val:  1   3   5   7  10  11  16  20  23  30  34  60

Binary search on idx 0..11.
To look up value at virtual index mid:

  row = mid // cols      (which row?)
  col = mid  % cols      (which column?)
  val = matrix[row][col]

Example: mid=5  cols=4
  row = 5//4 = 1
  col = 5%4  = 1
  matrix[1][1] = 11  ✓

The matrix is just a 1D sorted array wearing a grid costume.
```

## When To Use This Pattern

- When a 2D matrix is sorted row-by-row end-to-end,
  think **flatten to 1D via index math**
- When you need O(log(m×n)), think **one binary
  search, not m binary searches**
- When converting 1D index to 2D, think
  **row = mid // cols, col = mid % cols**
- When the matrix is sorted within rows but not
  across rows, think **search top-right corner
  instead** (different problem — LC 240)

## The Approach

Get the number of rows and columns, then treat the
entire matrix as a flat sorted array of length
rows × cols.
Run binary search from index 0 to rows×cols minus one.
At each step, convert the mid index to row and
column using integer division and modulo, then
compare that cell's value to the target.
Return True on a match, False if the loop exhausts.

In [3]:
from typing import List  # type hints for the solution

In [4]:
def test_harness(func):
    tests = [
        # (matrix, target, expected)
        ([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 3,  True),
        ([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 13, False),
        ([[1]],                                   1,  True),
        ([[1]],                                   2,  False),
        ([[1,1]],                                 1,  True),  # 1-row
        ([[1],[3],[5]],                            3,  True),  # 1-col
        ([[1],[3],[5]],                            2,  False),
        ([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 1,  True),
        ([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 60, True),
        ([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 0,  False),
    ]

    passed = 0
    for i, (matrix, target, expected) in enumerate(tests):
        result = func([r[:] for r in matrix], target)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"target={target} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [32]:
def searchMatrix(
    matrix: List[List[int]], target: int
) -> bool:
    """
    Return True if target is in the sorted 2D matrix.

    Treat matrix as a flat sorted array of m*n elements.
    Binary search lo=0..hi=m*n-1. At mid, convert to
    row=mid//cols, col=mid%cols. Compare matrix[row][col]
    to target; adjust lo/hi. Return True on match.

    Time:  O(log(m*n)) — one binary search
    Space: O(1) — two pointers only
    """
    rows = len(matrix)
    cols = len(matrix[0])
    length = rows * cols
    l, r = 0, length -1
    def valAt(offset):
        return matrix[offset//cols][offset%cols]
    while l <= r:
        mid = l + (r -l) // 2
        if valAt(mid) == target:
            return True
        if target >= valAt(mid):
            l = mid + 1
        else:
            r = mid -1
    return False
'''
True
False
True
True
Test 1: PASSED | target=3 | expected=True | got=True
Test 2: PASSED | target=13 | expected=False | got=False
Test 3: PASSED | target=1 | expected=True | got=True
Test 4: PASSED | target=2 | expected=False | got=False
Test 5: PASSED | target=1 | expected=True | got=True
Test 6: PASSED | target=3 | expected=True | got=True
Test 7: PASSED | target=2 | expected=False | got=False
Test 8: PASSED | target=1 | expected=True | got=True
Test 9: PASSED | target=60 | expected=True | got=True
Test 10: PASSED | target=0 | expected=False | got=False

10/10 tests passed

'''

# Quick debug — run this cell while building
m = [[1,3,5,7],[10,11,16,20],[23,30,34,60]]
print(searchMatrix(m, 3))   # True
print(searchMatrix(m, 13))  # False
print(searchMatrix(m, 1))   # True
print(searchMatrix(m, 60))  # True
test_harness(searchMatrix)

True
False
True
True
Test 1: PASSED | target=3 | expected=True | got=True
Test 2: PASSED | target=13 | expected=False | got=False
Test 3: PASSED | target=1 | expected=True | got=True
Test 4: PASSED | target=2 | expected=False | got=False
Test 5: PASSED | target=1 | expected=True | got=True
Test 6: PASSED | target=3 | expected=True | got=True
Test 7: PASSED | target=2 | expected=False | got=False
Test 8: PASSED | target=1 | expected=True | got=True
Test 9: PASSED | target=60 | expected=True | got=True
Test 10: PASSED | target=0 | expected=False | got=False

10/10 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(searchMatrix)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Linear scan every cell | O(m×n) | O(1) |
| Binary search per row | O(m + log n) | O(1) |
| Flatten + single binary search | O(log(m×n)) | O(1) |

Single binary search on the virtual 1D index is
optimal — O(log(m×n)) is better than O(m + log n)
when m is large.

## Real World Connection

At Citi, the telemetry summary is stored as a
server × day matrix sorted by server tier then
by date — exactly the structure this problem
describes.
Looking up whether a specific (server, date) pair
has a recorded metric uses the flatten-and-binary-
search approach: treat the grid as a sorted list
and locate the cell in O(log(m×n)).
On AWS Glue, partition discovery works the same
way: a 2D partition structure (region × date) is
searched with a binary index rather than a full
directory scan.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra

In [23]:
m = [[1,3,5,7],[10,11,16,20],[23,30,34,60]]
rows = 3 
cols = 4
print (f"0, 0   offset= 0 .. {m[0][0]} and the othwe way {0 % rows} and {0 // cols}")
print (3 //cols)
print ( 3 % cols)

0, 0   offset= 0 .. 1 and the othwe way 0 and 0
0
3
